In [1]:
import sys
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import DataLoader

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").exists() and ROOT_DIR.parent.exists():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

from song_recommender.paths import DATA_DIR, BASELINE_DIR, RESNET_MODELS_DIR, CONFIGS_DIR
from song_recommender.utils import load_config
from song_recommender.training import spec_baseline
from song_recommender.models import StemLateFusionResNet18, load_resnet_model, build_song_embeddings
from song_recommender.models.loader import load_tag_cluster_map, load_valid_tags
from song_recommender.data import StemSongDataset, TrackIndexer
from song_recommender.features.tag_features import add_tag_cluster_features
from song_recommender.evaluation import (
    topk_cosine, build_cluster_relevance_vector,
    precision_at_k, recall_at_k, average_precision_at_k,
    ndcg_at_k, jaccard_similarity, dominant_cluster_accuracy_at_k,
)

sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"device: {device}")

device: cuda


# Evaluation Metrics

We start this notebook by explaining each of the metrics that we will evaluate our models with. To start, we load the validation datset with the adjusted tag embeddings and clusters.

In [2]:
# Load config
config = load_config(CONFIGS_DIR / 'embeddings.yaml')

# Load the dataset files along with tag information
train_df = add_tag_cluster_features(pd.read_parquet(DATA_DIR / 'processed/train.parquet'), load_valid_tags(), load_tag_cluster_map())
val_df = add_tag_cluster_features(pd.read_parquet(DATA_DIR / 'processed/val.parquet'), load_valid_tags(), load_tag_cluster_map())

full_df = pd.concat([train_df, val_df], ignore_index = True)

full_df.sample(10)

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,deezer_artist,isrc,deezer_tags,deezer_year,tag_list,clean_tags,tag_set,tag_clusters,cluster_set,dominant_cluster
6412,TROLTIA128F4262730,In Her Blood,Converge,https://p.scdn.co/mp3-preview/3c920f281f4ddb22...,1eBSLJXzBZ6nB5PBOH4TKr,"death_metal, hardcore, metalcore",Metal,2004,246826,0.478,...,Converge,USEP40414112,alternative,2008,"[death_metal, hardcore, metalcore]","[death_metal, hardcore, metalcore]","{death_metal, hardcore, metalcore}","[3, 13, 3]","{3, 13}",3.0
1839,TRIXQTL128F92D3184,The Akara,Beirut,https://p.scdn.co/mp3-preview/a4064821bcae5cad...,08Fzdan1z0prqCZDoTJqzK,"indie, folk, singer_songwriter, american, 00s",NaN,2009,235760,0.623,...,Beirut,USCPT0910004,"alternative, rock, indie rock/rock pop",2009,"[indie, folk, singer_songwriter, american, 00s]","[indie, folk, singer_songwriter, american, 00s]","{singer_songwriter, american, 00s, folk, indie}","[18, 17, 17, 19, 19]","{17, 18, 19}",17.0
128,TRPSSGT12903CC23BD,Take It to the Limit,Eagles,https://p.scdn.co/mp3-preview/4a967da0de1ea40c...,0JLQwnFGBbM69Hn9LlkoAu,"rock, classic_rock, 70s",NaN,2003,287090,0.435,...,Eagles,USEE11300349,pop,2013,"[rock, classic_rock, 70s]","[classic_rock, 70s]","{classic_rock, 70s}","[16, 16]",{16},16.0
7962,TRMJUWQ128F4233CC2,I'm Going To Hell For This One,NOFX,https://p.scdn.co/mp3-preview/f3a0ebff3830f715...,1DNfXuvFas5z1Fjs4JJAw2,"punk, punk_rock, ska",Punk,2006,113533,0.404,...,NOFX,USFW41075818,"alternative, indie rock",2010,"[punk, punk_rock, ska]","[punk, punk_rock, ska]","{punk_rock, punk, ska}","[15, 15, 15]",{15},15.0
6342,TRVEXXG128F9333788,From These Wounds,All That Remains,https://p.scdn.co/mp3-preview/c258beecd1062717...,3CBubq98mfK0ydjPaH7gMI,"death_metal, metalcore, melodic_death_metal",Metal,2007,275800,0.273,...,All That Remains,USRZR0242002,metal,2007,"[death_metal, metalcore, melodic_death_metal]","[death_metal, metalcore, melodic_death_metal]","{melodic_death_metal, death_metal, metalcore}","[3, 3, 3]",{3},3.0
4337,TRKNBMT128F933A537,Der weinende Hadnur,Burzum,https://p.scdn.co/mp3-preview/1b2f35062d48c8ce...,68DBou7PMvbKZeukhojZau,"ambient, black_metal, dark_ambient",NaN,1999,76173,0.886,...,Burzum,GBXGC0819008,"electro, pop, rock, metal",2008,"[ambient, black_metal, dark_ambient]","[ambient, black_metal, dark_ambient]","{black_metal, dark_ambient, ambient}","[7, 1, 5]","{1, 5, 7}",7.0
2578,TRPYPDN128E0781E8D,Somebody Got Murdered,The Clash,https://p.scdn.co/mp3-preview/7a3054d06a74118d...,0iCAn6VQBfeSbd00RBVOdt,"punk, 80s, punk_rock",Reggae,1980,214066,0.487,...,The Clash,GBARL1200732,"alternative, indie rock, pop, international po...",1980,"[punk, 80s, punk_rock]","[punk, 80s, punk_rock]","{punk_rock, punk, 80s}","[15, 20, 15]","{20, 15}",15.0
8692,TRSGHAX128F4214DAD,Light Speed,Dr. Dre,https://p.scdn.co/mp3-preview/8ec6aaf03c14a215...,05Xjx4CoTYg2LyD3zGIGP7,"rap, hip_hop",Rap,1999,161093,0.861,...,Dr. Dre,USIR19915076,rap/hip hop,2008,"[rap, hip_hop]","[rap, hip_hop]","{hip_hop, rap}","[10, 10]",{10},10.0
5318,TRBTPDU128E07881EA,Your Silent Face,New Order,https://p.scdn.co/mp3-preview/d6ef8d3e1710135f...,09DCQgGsT2yc9DKR7uHPJG,"electronic, 80s, new_wave, post_punk, beautifu...",Rock,2017,382160,0.588,...,New Order,GBAAP1500383,"pop, rock",1999,"[electronic, 80s, new_wave, post_punk, beautif...","[electronic, 80s, new_wave, post_punk, beautif...","{80s, beautiful, electronic, new_wave, post_pu...","[11, 20, 4, 4, 18, 4]","{18, 11, 20, 4}",4.0
5571,TRNERXS128F423ED51,Cuts You Up,Peter Murphy,https://p.scdn.co/mp3-preview/c615d4831bba692c...,4ijNab1GM0JSZoe0I2KCOd,"alternative, 80s, new_wave, post_punk, gothic",Rock,1989,327133,0.630,...,Peter Murphy,GBAZP8900105,alternative,1989,"[alternative, 80s, new_wave, post_punk, gothic]","[alternative, 80s, new_wave, post_punk, gothic]","{80s, gothic, new_wave, alternative, post_punk}","[19, 20, 4, 4, 4]","{19, 20, 4}",4.0


## Relevance

Before defining each metric, we need to define what it means for a recommended track to be *relevant* to the query track. We will say that the recommended track is relevant to the query track if

1. the query and the recommendation have the same `dominant_cluster`, or
2. they have significant cluster overlap.

We say that the two tracks have *significant cluster overlap* if the [Jaccard similarity index](https://en.wikipedia.org/wiki/Jaccard_index) is greater than or equal to a chosen threshold, $\tau$. More precisely, if $Q$ is the **set** of clusters for the query and $R$ is the **set** of clusters for the recommended track, we say that the overlap is significant if 

$$J(Q, R) = \frac{\lvert Q\cap R \rvert}{\lvert Q\cup R \rvert} \ge \tau.$$

For this notebook, we chose $\tau = 0.25$.

**Remark:** recall that `dominant_cluster` is determined by the mode of `tag_clusters`. If `tag_clusters` is multimodal, then the first appearing cluster in the list of multimodal clusters is selected. So while repetition of clusters in `tag_clusters` is not represented in the Jaccard index, it is represented in `dominant_cluster`.

## Metric Functions

For each query, we assigned a relevance vector $\mathbf{r} = \left[r_1, r_2, \ldots, r_k\right]^T$ using the tag embeddings and clusters, where $r_i = 1$ if the $i$-th recommended track is relevant to the query track and 0 otherwise for each $1 \le i \le k$.

We consider the following metrics for a query and its relevance vector:

1. Precision@k: number of relevant neighbors out of $k$
    - Measures recommendation quality, i.e., accuracy of relevant retrieval
    - Not rank aware
2. Recall@k: number of relevant neighbors out of total number of relevant items in dataset
    - Measures models ability to retrieve all relevant tracks
    - Will be very small if there are many "relevant" tracks, especially for small $k$
3. AP@k (Average Precision at k): Average of $r_i \cdot $ precision@i for $i = 1,\ldots, k$
    - Measures ranking quality
    - Penalizes relevant tracks appearing later in $k$-neighbors
4. nDCG@k ([Normalized Discounted Cumulative Gain](https://www.google.com/url?sa=i&source=web&rct=j&url=https://en.wikipedia.org/wiki/Discounted_cumulative_gain) at k): DCG@k over IDCG@k (Ideal DCG)
    * DCG@k is the sum of relevance results, discounted logarithmically by their position in $k$-neighbors:

    $$\text{DCG}@k = \sum_{i = 1}^k \frac{r_i}{\log_2(i+1)}$$
    * IDCG@k is the maximum possible DCG if $k$-neighbors were sorted by relevance, i.e., if $0 \le t \le k$ and $t$ of the $k$-neighbors are relevant, then $r_i > 0$ for all $i \le t$ and $$\text{IDCG}@k = \sum_{i = 1}^t \frac{r_i}{\log_2(i+1)}$$
    - Measures ranking quality
    - Rewards highly relevant tracks appearing earlier in $k$-neigbors
    - Grades relevance, e.g., highly relevant vs. partially relevant
    - Not as useful if relevance has a binary grading
    - Normalized for comparision across queries
5. Tag Jaccard Similarity: Average of tag Jaccard similarity for query against each $k$-neighbor
    - Measures if $k$-neighbors share tags
    - Not using relevancy vectors
    - Not used to compute relevancy
6. Cluster Jaccard Similarity: Average of cluster Jaccard similarity for query against each $k$-neighbor
    - Measures if $k$-neighbors share clusters
    - Not using relevancy vectors
    - Used to compute relevancy
7. Dominant Cluster Accuracy@k: number of neighbors whose dominant cluster matches query out of $k$

In [3]:
def evaluate_embeddings(
    query_emb, db_emb, query_df, db_df,
    k_list=(1, 5, 10, 20), overlap_threshold=0.25, self_retrieval=False,
):
    """
    Evaluate embedding retrieval quality using tag-cluster metrics.

    Args:
        query_emb:        (N_q, D) normalized embeddings for queries
        db_emb:           (N_db, D) normalized embeddings for database
        query_df/db_df:   DataFrames with cluster_set, dominant_cluster, tag_set columns
        k_list:           k values to evaluate
        overlap_threshold: Jaccard threshold for cluster overlap relevance
        self_retrieval:   if True, exclude index i from db (when query_emb is db_emb)

    Returns:
        dict {k: {metric_name: float}}
    """
    db_cluster_sets = db_df["cluster_set"].values
    db_dom_clusters = db_df["dominant_cluster"].values
    db_tag_sets     = db_df["tag_set"].values
    q_cluster_sets  = query_df["cluster_set"].values
    q_dom_clusters  = query_df["dominant_cluster"].values
    q_tag_sets      = query_df["tag_set"].values

    query_ids = query_df["spotify_id"].tolist()
    db_id_to_idx = {sid: i for i, sid in enumerate(db_df["spotify_id"].tolist())}

    accumulators = {k: {
        "prec": [], "rec": [], "ap": [], "ndcg": [],
        "tag_j": [], "clus_j": [], "dom_acc": [],
    } for k in k_list}

    for i in range(len(query_emb)):
        exclude = db_id_to_idx.get(query_ids[i]) if self_retrieval else None

        full_rel = build_cluster_relevance_vector(
            q_cluster_sets[i], db_cluster_sets,
            q_dom_clusters[i], db_dom_clusters,
            overlap_threshold,
        )
        total_relevant = max(0, int(np.sum(full_rel)) - (1 if self_retrieval else 0))

        for k in k_list:
            idx, _ = topk_cosine(db_emb, query_emb[i], k, exclude_idx=exclude)

            relevance = build_cluster_relevance_vector(
                q_cluster_sets[i], db_cluster_sets[idx],
                q_dom_clusters[i], db_dom_clusters[idx],
                overlap_threshold,
            )

            acc = accumulators[k]
            acc["prec"].append(precision_at_k(relevance))
            acc["rec"].append(recall_at_k(relevance, total_relevant))
            acc["ap"].append(average_precision_at_k(relevance))
            acc["ndcg"].append(ndcg_at_k(relevance))
            acc["dom_acc"].append(dominant_cluster_accuracy_at_k(q_dom_clusters[i], db_dom_clusters[idx]))
            acc["tag_j"].append(np.mean([jaccard_similarity(q_tag_sets[i], t) for t in db_tag_sets[idx]]))
            acc["clus_j"].append(np.mean([jaccard_similarity(q_cluster_sets[i], c) for c in db_cluster_sets[idx]]))

    return {
        k: {
            "precision@k":            np.mean(acc["prec"]),
            "recall@k":               np.mean(acc["rec"]),
            "map@k":                  np.mean(acc["ap"]),
            "ndcg@k":                 np.mean(acc["ndcg"]),
            "tag_jaccard":            np.mean(acc["tag_j"]),
            "cluster_jaccard":        np.mean(acc["clus_j"]),
            "dominant_cluster_acc@k": np.mean(acc["dom_acc"]),
        }
        for k, acc in accumulators.items()
    }

# Embedding Computer

In [4]:
# Baseline Model Information
baseline_emb = spec_baseline.build_embeddings(full_df, config)

# Sanity Checking
print("baseline:", baseline_emb.shape)

# Load Train + Validation Data
loader = DataLoader(StemSongDataset(full_df, image_size=224), batch_size=32, shuffle=False, num_workers=0)

# Resnet Notebook04 Information
model  = load_resnet_model( RESNET_MODELS_DIR / "04_Resnet18_contrastive_tags/checkpoint.pt", device)
out    = build_song_embeddings(model, loader, device)
resnet04_emb = out["song_embeddings"]

print("resnet04:", resnet04_emb.shape)

# Resnet Notebook05 Information
model  = load_resnet_model( RESNET_MODELS_DIR / "05_Resnet18_contrastive_tags_infonce/checkpoint.pt", device)
out    = build_song_embeddings(model, loader, device)
resnet05_emb = out["song_embeddings"]

print("resnet05:", resnet05_emb.shape)

# Resnet Notebook04_BEEG Information
model  = load_resnet_model( RESNET_MODELS_DIR / "04a_Resnet18_contrastive_tags_BEEG/checkpoint.pt", device)
out    = build_song_embeddings(model, loader, device)
resnet04a_emb = out["song_embeddings"]

print("resnet04a:", resnet04a_emb.shape)

# Resnet Notebook06 Information
model  = load_resnet_model( RESNET_MODELS_DIR / "06_Resnet18_contrastive_tags_audio_grounded_infonce/checkpoint.pt", device)
out    = build_song_embeddings(model, loader, device)
resnet06_emb = out["song_embeddings"]

print("resnet06:", resnet06_emb.shape)

# Future Resnet Models...

# Sanity Checking
print("baseline:", baseline_emb.shape)
print("resnet04:", resnet04_emb.shape)
print("resnet05:", resnet05_emb.shape)
print("resnet04a:", resnet04a_emb.shape)
print("resnet06:", resnet06_emb.shape)

embedding 1/9549
embedding 201/9549
embedding 401/9549
embedding 601/9549
embedding 801/9549
embedding 1001/9549
embedding 1201/9549
embedding 1401/9549
embedding 1601/9549
embedding 1801/9549
embedding 2001/9549
embedding 2201/9549
embedding 2401/9549
embedding 2601/9549
embedding 2801/9549
embedding 3001/9549
embedding 3201/9549
embedding 3401/9549
embedding 3601/9549
embedding 3801/9549
embedding 4001/9549
embedding 4201/9549
embedding 4401/9549
embedding 4601/9549
embedding 4801/9549
embedding 5001/9549
embedding 5201/9549
embedding 5401/9549
embedding 5601/9549
embedding 5801/9549
embedding 6001/9549
embedding 6201/9549
embedding 6401/9549
embedding 6601/9549
embedding 6801/9549
embedding 7001/9549
embedding 7201/9549
embedding 7401/9549
embedding 7601/9549
embedding 7801/9549
embedding 8001/9549
embedding 8201/9549
embedding 8401/9549
embedding 8601/9549
embedding 8801/9549
embedding 9001/9549
embedding 9201/9549
embedding 9401/9549
baseline: (9549, 5120)
resnet04: (9549, 64)
res

In [5]:
# Load Train + Validation Data
loader = DataLoader(StemSongDataset(full_df, image_size=224), batch_size=32, shuffle=False, num_workers=0)

# Resnet Notebook07 Information
model  = load_resnet_model( RESNET_MODELS_DIR / "07_Resnet18_audio_centric_blended_teacher/checkpoint.pt", device)
out    = build_song_embeddings(model, loader, device)
resnet07_emb = out["song_embeddings"]

In [9]:
print("resnet07:", resnet07_emb.shape)

resnet07: (9549, 64)


## Embedding Loading

In [8]:
data = np.load(DATA_DIR / "processed" / "model_runs" / "all_model_embeddings.npz")

baseline_emb  = data["baseline"]
resnet04_emb  = data["resnet04"]
resnet05_emb  = data["resnet05"]
resnet04a_emb = data["resnet04a"]
resnet06_emb  = data["resnet06"]

print("baseline:",  baseline_emb.shape)
print("resnet04:",  resnet04_emb.shape)
print("resnet05:",  resnet05_emb.shape)
print("resnet04a:", resnet04a_emb.shape)
print("resnet06:",  resnet06_emb.shape)

baseline: (9549, 5120)
resnet04: (9549, 64)
resnet05: (9549, 64)
resnet04a: (9549, 64)
resnet06: (9549, 64)


## Metric Evaluations

In [10]:
k_list = [1, 5, 10, 20, 50]

val_mask = full_df["spotify_id"].isin(val_df["spotify_id"]).values

models = {
    "Baseline":               baseline_emb,
    "ResNet04 (Contrastive)": resnet04_emb,
    "ResNet04_BEEG (Contrastive)": resnet04a_emb,
    "ResNet05 (InfoNCE)":     resnet05_emb,
    "ResNet06": resnet06_emb,
    "ResNet07": resnet07_emb,
    # Future Models to be added...
}

all_results = {}
for name, emb in models.items():
    print(f"evaluating {name}...")
    all_results[name] = evaluate_embeddings(
        query_emb=emb[val_mask], db_emb=emb,
        query_df=val_df, db_df=full_df,
        k_list=k_list, overlap_threshold=0.25, self_retrieval=True,
    )
print("done.")

evaluating Baseline...
evaluating ResNet04 (Contrastive)...
evaluating ResNet04_BEEG (Contrastive)...
evaluating ResNet05 (InfoNCE)...
evaluating ResNet06...
evaluating ResNet07...
done.


In [11]:
k = 1
display(pd.DataFrame({name: res[k] for name, res in all_results.items()}).T.round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
Baseline,0.4223,0.0003,0.4223,0.4223,0.1452,0.2317,0.2440
ResNet04 (Contrastive),0.6869,0.0004,0.6869,0.6869,0.2216,0.3538,0.4050
ResNet04_BEEG (Contrastive),0.6641,0.0004,0.6641,0.6641,0.2260,0.3551,0.3849
ResNet05 (InfoNCE),0.6704,0.0004,0.6704,0.6704,0.2230,0.3487,0.3891
ResNet06,0.6282,0.0004,0.6282,0.6282,0.2251,0.3455,0.3753
ResNet07,0.5529,0.0003,0.5529,0.5529,0.2062,0.3055,0.3338


In [12]:
k = 5
display(pd.DataFrame({name: res[k] for name, res in all_results.items()}).T.round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
Baseline,0.4040,0.0012,0.5491,0.6314,0.1303,0.2172,0.2229
ResNet04 (Contrastive),0.6749,0.0021,0.7564,0.8086,0.2198,0.3548,0.3916
ResNet04_BEEG (Contrastive),0.6648,0.0020,0.7429,0.7979,0.2248,0.3551,0.3779
ResNet05 (InfoNCE),0.6571,0.0020,0.7464,0.7994,0.2202,0.3488,0.3852
ResNet06,0.6267,0.0019,0.7173,0.7770,0.2120,0.3337,0.3636
ResNet07,0.5364,0.0016,0.6533,0.7265,0.1944,0.2973,0.3160


In [13]:
k = 10
display(pd.DataFrame({name: res[k] for name, res in all_results.items()}).T.round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
Baseline,0.3907,0.0024,0.5210,0.6566,0.1209,0.2073,0.2109
ResNet04 (Contrastive),0.6720,0.0042,0.7405,0.8222,0.2214,0.3558,0.3900
ResNet04_BEEG (Contrastive),0.6611,0.0041,0.7257,0.8104,0.2217,0.3518,0.3806
ResNet05 (InfoNCE),0.6540,0.0040,0.7234,0.8090,0.2184,0.3476,0.3853
ResNet06,0.6273,0.0039,0.6970,0.7909,0.2105,0.3348,0.3670
ResNet07,0.5249,0.0032,0.6247,0.7414,0.1881,0.2894,0.3029


In [14]:
k = 20
display(pd.DataFrame({name: res[k] for name, res in all_results.items()}).T.round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
Baseline,0.3743,0.0046,0.4776,0.6671,0.1117,0.1963,0.1998
ResNet04 (Contrastive),0.6713,0.0083,0.7177,0.8288,0.2208,0.3546,0.3915
ResNet04_BEEG (Contrastive),0.6546,0.0081,0.7042,0.8178,0.2166,0.3481,0.3781
ResNet05 (InfoNCE),0.6511,0.0080,0.6996,0.8156,0.2146,0.3460,0.3811
ResNet06,0.6196,0.0076,0.6739,0.8009,0.2038,0.3296,0.3588
ResNet07,0.5134,0.0063,0.5897,0.7511,0.1807,0.2828,0.2944


In [15]:
k = 50
display(pd.DataFrame({name: res[k] for name, res in all_results.items()}).T.round(4))

,precision@k,recall@k,map@k,ndcg@k,tag_jaccard,cluster_jaccard,dominant_cluster_acc@k
Baseline,0.3569,0.0109,0.4238,0.6806,0.1029,0.1857,0.1870
ResNet04 (Contrastive),0.6709,0.0207,0.6982,0.8414,0.2195,0.3537,0.3932
ResNet04_BEEG (Contrastive),0.6479,0.0199,0.6788,0.8293,0.2111,0.3435,0.3742
ResNet05 (InfoNCE),0.6496,0.0199,0.6767,0.8275,0.2115,0.3444,0.3773
ResNet06,0.6101,0.0187,0.6464,0.8139,0.1973,0.3232,0.3516
ResNet07,0.5033,0.0153,0.5505,0.7639,0.1719,0.2736,0.2853


In [15]:
np.savez(
    DATA_DIR / "processed" / "model_runs" / "all_model_embeddings.npz",
    baseline=baseline_emb,
    resnet04=resnet04_emb,
    resnet05=resnet05_emb,
    resnet04a=resnet04a_emb,
    resnet06=resnet06_emb,
)

In [16]:
data = np.load(DATA_DIR / "processed" / "model_runs" / "all_model_embeddings.npz")